# 01 数据加载与清洗

**目的**：确认数据可信，建立数据质量基线。每个清洗决策都要能回答「为什么这样做」。

In [ ]:
import pandas as pd
import sys
sys.path.append('..')
from utils.db import load_csv_to_sqlite, query

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. 加载数据

In [ ]:
df = pd.read_csv('../data/user_behavior.csv')
print(f'总行数: {len(df):,}')
print(f'总列数: {len(df.columns)}')
print(f'\n列名和类型:\n{df.dtypes}')
df.head(10)

## 2. 数据质量检查

In [ ]:
print('=== 缺失值 ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'缺失数': missing, '缺失率(%)': missing_pct}))

In [ ]:
print('=== 重复行 ===')
dup_rows = df.duplicated().sum()
print(f'完全重复行数: {dup_rows} ({dup_rows/len(df)*100:.2f}%)')

In [ ]:
print('=== 用户-商品-时间 三重重复（可能是数据采集重复上报）===')
subset_cols = ['user_id', 'item_id', 'behavior_type', 'time']
dup_subset = df.duplicated(subset=subset_cols).sum()
print(f'user_id + item_id + behavior_type + time 重复: {dup_subset}')
if dup_subset > 0:
    print('\n示例:')
    display(df[df.duplicated(subset=subset_cols, keep=False)].head(10))

## 3. 基础统计

In [ ]:
print(f'唯一用户数: {df["user_id"].nunique():,}')
print(f'唯一商品数: {df["item_id"].nunique():,}')
print(f'唯一品类数: {df["category_id"].nunique():,}')
print(f'\n时间范围: {df["time"].min()} ~ {df["time"].max()}')

In [ ]:
print('=== 行为类型分布 ===')
behavior_dist = df['behavior_type'].value_counts()
behavior_pct = df['behavior_type'].value_counts(normalize=True).mul(100).round(2)
print(pd.DataFrame({'数量': behavior_dist, '占比(%)': behavior_pct}))

In [ ]:
print('=== 用户活跃度分布（Top 20 用户）===')
user_activity = df.groupby('user_id').size().sort_values(ascending=False)
print(user_activity.head(20))
print(f'\n人均行为数: {len(df) / df["user_id"].nunique():.1f}')
print(f'中位数: {user_activity.median():.0f}')

## 4. 时间字段处理

In [ ]:
# 转换为 datetime 类型
df['time'] = pd.to_datetime(df['time'])

# 提取时间维度（后续分析用）
df['date'] = df['time'].dt.date
df['hour'] = df['time'].dt.hour
df['weekday'] = df['time'].dt.dayofweek  # 0=周一
df['is_weekend'] = df['weekday'].isin([5, 6])

print('时间字段处理完成')
print(f'日期范围: {df["date"].min()} ~ {df["date"].max()}')
print(f'\n每日数据量:\n{df["date"].value_counts().sort_index()}')

## 5. 数据质量报告

In [ ]:
print('=' * 50)
print('数据质量报告')
print('=' * 50)

checks = {
    '总行数': f'{len(df):,}',
    '总列数': len(df.columns),
    '唯一用户数': f'{df["user_id"].nunique():,}',
    '唯一商品数': f'{df["item_id"].nunique():,}',
    '唯一品类数': f'{df["category_id"].nunique():,}',
    '缺失值总数': df.isnull().sum().sum(),
    '重复行(完全)': dup_rows,
    '时间范围': f'{df["time"].min()} ~ {df["time"].max()}',
    '行为类型': f'pv {behavior_pct["pv"]}% | cart {behavior_pct["cart"]}% | fav {behavior_pct["fav"]}% | buy {behavior_pct["buy"]}%',
    '人均行为数': f'{len(df) / df["user_id"].nunique():.1f}',
    '核心结论': '数据完整无缺失，分布合理，可直接进入分析',
}

for k, v in checks.items():
    print(f'  {k}: {v}')

## 6. 导出清洗后数据

In [ ]:
# 去重后保存到 SQLite
df_clean = df.drop_duplicates(subset=['user_id', 'item_id', 'behavior_type', 'time'])
print(f'清洗前: {len(df):,} 行')
print(f'去重后: {len(df_clean):,} 行')
print(f'移除: {len(df) - len(df_clean):,} 行')

# 写入 SQLite（用于纯 SQL 分析）
import sqlite3
conn = sqlite3.connect('../data/taobao_behavior.db')
df_clean.to_sql('user_behavior_clean', conn, if_exists='replace', index=False)
conn.close()
print('\n已写入 SQLite 表: user_behavior_clean')